# InequalityLens — Exploratory Data Analysis

This notebook explores global inequality data from the World Bank, US Census, and WHO.

**Contents:**
1. Data loading + overview
2. Distribution analysis (Gini, GDP, life expectancy)
3. Correlation heatmap
4. Lorenz curve computation
5. Clustering validation
6. Time-series trends
7. US state-level analysis

In [ ]:
import sys
sys.path.insert(0, '../data')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from data_pipeline import run_pipeline, compute_lorenz, gini_from_lorenz

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

print('Libraries loaded ✓')

In [ ]:
# Run pipeline (fetches data or uses synthetic fallback)
data = run_pipeline()
world = data['world']
gini_ts = data['gini_ts']
us_states = data['us_states']
lorenz = data['lorenz']

print(f'World dataset: {world.shape}')
print(f'Gini time series: {gini_ts.shape}')
print(f'US states: {us_states.shape}')
world.describe()

In [ ]:
# 1. Distribution plots
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Distribution of socioeconomic indicators', fontsize=14, fontweight='bold')

cols = ['gini_index','gdp_per_capita','life_expectancy','poverty_rate','adult_literacy','health_expenditure_pc']
for ax, col in zip(axes.flat, cols):
    if col in world.columns:
        data_clean = world[col].dropna()
        ax.hist(data_clean, bins=25, edgecolor='white', color='#6366f1', alpha=0.85)
        ax.axvline(data_clean.mean(), color='#ef4444', linestyle='--', label=f'Mean: {data_clean.mean():.1f}')
        ax.axvline(data_clean.median(), color='#10b981', linestyle='--', label=f'Median: {data_clean.median():.1f}')
        ax.set_title(col.replace('_',' ').title())
        ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('../data/processed/distributions.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# 2. Correlation heatmap
numeric_cols = ['gini_index','gdp_per_capita','life_expectancy','poverty_rate','adult_literacy','health_expenditure_pc']
corr = world[numeric_cols].dropna().corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn_r',
            center=0, ax=ax, square=True, linewidths=0.5)
ax.set_title('Correlation matrix — inequality indicators', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/processed/correlation_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# 3. GDP vs Gini scatter (Kuznets curve)
fig, ax = plt.subplots(figsize=(10, 7))

plot_df = world[['country_name','gini_index','gdp_per_capita','life_expectancy']].dropna()
scatter = ax.scatter(
    plot_df['gdp_per_capita'], plot_df['gini_index'],
    c=plot_df['life_expectancy'], cmap='RdYlGn',
    s=70, alpha=0.7, edgecolors='white', linewidth=0.5
)
for _, row in plot_df.nlargest(5,'gini_index').iterrows():
    ax.annotate(row['country_name'], (row['gdp_per_capita'], row['gini_index']),
                fontsize=8, ha='left', va='bottom', alpha=0.8)

plt.colorbar(scatter, ax=ax, label='Life expectancy')
ax.set_xlabel('GDP per capita (USD)')
ax.set_ylabel('Gini index')
ax.set_title('GDP per capita vs Gini index (colour = life expectancy)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/processed/gdp_vs_gini.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# 4. Lorenz curves
fig, ax = plt.subplots(figsize=(9, 7))
colors = ['#6366f1','#10b981','#f59e0b','#ef4444','#8b5cf6','#06b6d4','#84cc16','#f97316']

for (country, group), color in zip(lorenz.groupby('country'), colors):
    ax.plot(group['population_share'], group['income_share'],
            label=f"{country} (Gini={group['gini'].iloc[0]:.3f})",
            linewidth=2, color=color)

ax.plot([0,1],[0,1], 'k--', alpha=0.3, label='Perfect equality')
ax.fill_between([0,1],[0,1],[0,0], alpha=0.04, color='gray')
ax.set_xlabel('Cumulative population share')
ax.set_ylabel('Cumulative income share')
ax.set_title('Lorenz curves — selected countries', fontsize=12, fontweight='bold')
ax.legend(fontsize=9, loc='upper left')
ax.set_xlim(0,1); ax.set_ylim(0,1)
plt.tight_layout()
plt.savefig('../data/processed/lorenz_curves.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# 5. Gini time series trends
fig, ax = plt.subplots(figsize=(12, 6))

key_countries = ['USA','DEU','BRA','SWE','ZAF','IND']
for code in key_countries:
    ts = gini_ts[gini_ts['country_code']==code].sort_values('year')
    if not ts.empty:
        ax.plot(ts['year'], ts['gini'], marker='o', markersize=3,
                label=ts['country_name'].iloc[0], linewidth=2)

ax.set_xlabel('Year')
ax.set_ylabel('Gini index')
ax.set_title('Gini index trends — key countries', fontsize=12, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('../data/processed/gini_trends.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# 6. US state inequality
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar: income by state (top/bottom 15)
top15 = us_states.nlargest(15, 'median_household_income')
bottom15 = us_states.nsmallest(15, 'median_household_income')

for ax, df, title, color in [
    (axes[0], top15, 'Top 15 states by median income', '#10b981'),
    (axes[1], bottom15, 'Bottom 15 states by median income', '#ef4444')
]:
    ax.barh(df['state'], df['median_household_income'], color=color, alpha=0.85)
    ax.set_xlabel('Median household income (USD)')
    ax.set_title(title, fontweight='bold')
    ax.invert_yaxis()
    for bar, val in zip(ax.patches, df['median_household_income']):
        ax.text(bar.get_width()+200, bar.get_y()+bar.get_height()/2,
                f'${val:,.0f}', va='center', fontsize=8)

plt.tight_layout()
plt.savefig('../data/processed/us_states_income.png', dpi=120, bbox_inches='tight')
plt.show()

print('\n=== EDA complete ===')
print(f'All charts saved to data/processed/')